In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("titanic.csv")
X = df.drop("survived", axis=1)
y = df["survived"]

numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categorical: ['sex', 'embarked']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),      # scale the numeric columns
    ("cat", OneHotEncoder(), categorical_features),   # one-hot the categorical columns
])

print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['pclass', 'age', 'sibsp', 'parch', 'fare']),
                                ('cat', OneHotEncoder(), ['sex', 'embarked'])])


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

print(pipe)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['pclass', 'age', 'sibsp',
                                                   'parch', 'fare']),
                                                 ('cat', OneHotEncoder(),
                                                  ['sex', 'embarked'])])),
                ('classifier', LogisticRegression(max_iter=1000))])


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe.fit(X_train, y_train)          # preprocessa E treina, tudo numa linha
predictions = pipe.predict(X_test)  # preprocessa o teste E prevê

print(f"Accuracy: {accuracy_score(y_test, predictions):.2%}")

Accuracy: 80.90%


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
print("Fold scores:", np.round(scores, 3))
print(f"Mean: {scores.mean():.2%}")

Fold scores: [0.769 0.796 0.789 0.782 0.81 ]
Mean: 78.91%


In [ ]:
#Cell6
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.2%}")
print(f"Test accuracy:    {grid.score(X_test, y_test):.2%}")